# ReqRE Demo

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve()))


from pathlib import Path

import networkx as nx

from reqre.cypher import rule_to_cypher
from reqre.gaphor_requirements import (
    load_requirement_relationships_from_file,
    load_requirements_from_file,
    push_requirement_relationships_to_neo4j,
    push_requirements_to_neo4j,
)
from reqre.graphml import export_rule_graphml
from reqre.neo4j import Neo4jClient
from reqre.rules import DpoRule, add_edge, add_node


ModuleNotFoundError: No module named 'reqre'

## 1. Push requirements from Gaphor to Neo4j\n

In [ ]:
repo_root = Path.cwd().resolve()
gaphor_path = repo_root / 'gaphor_files' / 'ArchBridgeRequirements.gaphor'

requirements = load_requirements_from_file(gaphor_path)
relationships = load_requirement_relationships_from_file(gaphor_path)

with Neo4jClient() as client:
    req_total = push_requirements_to_neo4j(client, requirements)
    rel_total = push_requirement_relationships_to_neo4j(client, relationships)

print(f'Pushed {req_total} requirements and {rel_total} relationships.')


## 2. Create the DPO rule\n

In [ ]:
def find_requirement(requirements, external_id, name):
    for req in requirements:
        if req.external_id == external_id and req.name == name:
            return req
    for req in requirements:
        if req.external_id == external_id:
            return req
    raise ValueError(f'Requirement {external_id} not found')

def gh_path(root, filename):
    path = root / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing Grasshopper file: {path}')
    return str(Path('gh') / filename)

def add_building_element(graph, node_id, name, gh_file, index=None):
    props = {'name': name, 'gh_file': gh_file}
    if index is not None:
        props['index'] = index
    add_node(graph, node_id, label='BuildingElement', props=props)

gh_root = repo_root / 'gh'
bridge_req = find_requirement(requirements, '0', 'Bridge')
arch_req = find_requirement(requirements, '1', 'Arch Bridge')

left = nx.MultiDiGraph()
interface = nx.MultiDiGraph()
right = nx.MultiDiGraph()

for req in (bridge_req, arch_req):
    props = {
        'external_id': req.external_id,
        'name': req.name,
        'text': req.text,
    }
    add_node(left, req.gaphor_id, label='Requirement', props=props)
    add_node(interface, req.gaphor_id, label='Requirement', props=props)
    add_node(right, req.gaphor_id, label='Requirement', props=props)

bridge_element_id = 'bridge_element'
arch_element_id = 'arch_element'
abutment_a_id = 'abutment_a'
abutment_b_id = 'abutment_b'
girder_id = 'girder'

add_building_element(
    right,
    bridge_element_id,
    'Bridge Element',
    gh_path(gh_root, 'twisty.gh'),
)
add_building_element(
    right,
    arch_element_id,
    'Arch Element',
    gh_path(gh_root, 'ArchSegment_Connector.gh'),
)
add_building_element(
    right,
    abutment_a_id,
    'Abutment A',
    gh_path(gh_root, 'Curb.gh'),
)
add_building_element(
    right,
    abutment_b_id,
    'Abutment B',
    gh_path(gh_root, 'Sidewalk.gh'),
)
add_building_element(
    right,
    girder_id,
    'Girder',
    gh_path(gh_root, 'Girder.gh'),
)

for index in range(1, 6):
    add_building_element(
        right,
        f'arch_segment_{index}',
        'Arch Segment',
        gh_path(gh_root, 'ArchSegment.gh'),
        index=index,
    )

for index in range(1, 5):
    add_building_element(
        right,
        f'spandrel_column_{index}',
        'Spandrel Column',
        gh_path(gh_root, 'Spandrel.gh'),
        index=index,
    )

add_edge(right, bridge_req.gaphor_id, bridge_element_id, rel_type='SATISFIES')
add_edge(right, arch_req.gaphor_id, arch_element_id, rel_type='SATISFIES')

add_edge(right, bridge_element_id, arch_element_id, rel_type='DECOMPOSES')
add_edge(right, bridge_element_id, abutment_a_id, rel_type='DECOMPOSES')
add_edge(right, bridge_element_id, abutment_b_id, rel_type='DECOMPOSES')
add_edge(right, bridge_element_id, girder_id, rel_type='DECOMPOSES')

for index in range(1, 6):
    add_edge(right, arch_element_id, f'arch_segment_{index}', rel_type='DECOMPOSES')

for index in range(1, 5):
    add_edge(right, arch_element_id, f'spandrel_column_{index}', rel_type='DECOMPOSES')

rule = DpoRule(left=left, interface=interface, right=right)
rule.summary()


## 3. Serialize the rule to GraphML\n

In [ ]:
graphml_path = repo_root / 'arch_bridge_rule.graphml'
export_rule_graphml(rule, graphml_path)
graphml_path


## 4. Serialize the rule to Cypher\n

In [ ]:
cypher = rule_to_cypher(rule)
print(cypher.query)
cypher.params


## 5. Push the Cypher to Neo4j\n

In [ ]:
with Neo4jClient() as client:
    client.execute(cypher.query, cypher.params)

print('Cypher pushed to Neo4j.')
